# 01 · Fruit-level partitioning (no leakage)
Splits **physical fruits** into train/val/test and verifies that no scan/day of a test fruit appears in train or val. Also builds repeated group-CV folds.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()/'nbpkg'))
import numpy as np, pandas as pd
from config import CFG
import dataset as ds, eval_core as ec
CFG.out_dir.mkdir(parents=True, exist_ok=True)
print('data_root :', CFG.data_root); print('fruit_key :', CFG.fruit_key)

In [ ]:
fruits = ds.build_fruit_index(CFG.data_root, fruit_key=CFG.fruit_key)[0]
by={f.fruit_id:f for f in fruits}
ids=[f.fruit_id for f in fruits]; labels=[f.label for f in fruits]

## Step 1 — stratified three-way split of fruits

In [ ]:
tr,va,te = ec.make_three_way_split(ids, labels, test_frac=CFG.test_frac,
        val_frac_of_trainval=CFG.val_frac_of_trainval, seed=CFG.seed)
ec.assert_no_leakage(('train',tr),('val',va),('test',te))
print(f'fruits  train={len(tr)} val={len(va)} test={len(te)}')

## Step 2 — verify NO scan/day of a test fruit leaks into train/val

In [ ]:
def scans(part): return {(i,s.day) for i in part for s in by[i].scans}
assert not (scans(tr)&scans(te)) and not (scans(va)&scans(te))
print('scan-level leakage check: PASSED')

## Step 3 — composition in BOTH fruits and scans (reviewer comments 2 & 3)

In [ ]:
def compo(name,part):
    inf=sum(by[i].label for i in part); ns=sum(len(by[i].scans) for i in part)
    return {'partition':name,'fruits':len(part),'infested_fruits':inf,
            'control_fruits':len(part)-inf,'scans':ns}
tbl=pd.DataFrame([compo('train',tr),compo('val',va),compo('test',te)])
display(tbl); tbl.to_csv(CFG.out_dir/'partition_composition.csv',index=False)

## Step 4 — persist split

In [ ]:
m=({i:'train' for i in tr}|{i:'val' for i in va}|{i:'test' for i in te})
pd.Series(m,name='split').rename_axis('fruit_id').reset_index()\
  .to_csv(CFG.out_dir/'split_single.csv',index=False)
print('saved split_single.csv')

## Step 5 — repeated stratified group CV (by fruit)

In [ ]:
rows=[]
for rep,trf,tef in ec.repeated_group_folds(ids,labels,n_splits=CFG.cv_n_splits,
        n_repeats=CFG.cv_n_repeats,seed=CFG.seed):
    ec.assert_no_leakage(('tr',trf),('te',tef))
    for i in tef: rows.append({'repeat':rep,'fruit_id':i})
pd.DataFrame(rows).to_csv(CFG.out_dir/'cv_folds.csv',index=False)
print('saved cv_folds.csv — every fold leakage-free by fruit')